# Week 6 — Integrative Capstone Project and Evaluation
## Wine Classification and Unsupervised Segmentation

**Complete pipeline:** data acquisition → cleaning → EDA → supervised modeling → hyperparameter analysis → evaluation → unsupervised clustering → insights → recommendations.

This notebook is designed as the reproducible technical companion to the professional Week 6 report.

## 1. Data Acquisition and Quality Checks

In [1]:
from sklearn.datasets import load_wine
import pandas as pd

wine = load_wine(as_frame=True)
df = wine.frame.copy()
df["target_name"] = df["target"].map(dict(enumerate(wine.target_names)))

print("Shape:", df.shape)
print("Classes:", list(wine.target_names))
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

Shape: (178, 15)
Classes: [np.str_('class_0'), np.str_('class_1'), np.str_('class_2')]
Missing values: 0
Duplicate rows: 0


## 2. Cleaning, Stratified Split and Leakage Control

In [2]:
from sklearn.model_selection import train_test_split

df = df.drop_duplicates()
X = df[wine.feature_names]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training:", X_train.shape)
print("Test:", X_test.shape)

Training: (142, 13)
Test: (36, 13)


## 3. Exploratory Data Analysis

In [3]:
print(df["target_name"].value_counts().sort_index())
display(df[wine.feature_names].describe().round(2))

target_name
class_0    59
class_1    71
class_2    48

       alcohol  malic_acid     ash  alcalinity_of_ash  magnesium  total_phenols  flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity     hue  od280/od315_of_diluted_wines  proline
count   178.00      178.00  178.00             178.00     178.00         178.00      178.00                178.00           178.00           178.00  178.00                        178.00   178.00
mean     13.00        2.34    2.37              19.49      99.74           2.30        2.03                  0.36             1.59             5.06    0.96                          2.61   746.89
std       0.81        1.12    0.27               3.34      14.28           0.63        1.00                  0.12             0.57             2.32    0.23                          0.71   314.91
min      11.03        0.74    1.36              10.60      70.00           0.98        0.34                  0.13             0.41             1.28    0.48          

## 4. Supervised Model Comparison

Three models with different inductive biases are compared. Scaling is inside the Logistic Regression and SVM pipelines to prevent preprocessing leakage during validation.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, random_state=42))
    ]),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", C=1.0, gamma="scale"))
    ])
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, average="weighted"),
        "Recall": recall_score(y_test, pred, average="weighted"),
        "F1": f1_score(y_test, pred, average="weighted"),
        "CV Mean": cross_val_score(model, X, y, cv=5, scoring="accuracy").mean()
    })

results = pd.DataFrame(rows).sort_values("F1", ascending=False)
display(results.round(4))

              Model  Accuracy  Precision  Recall    F1  CV Mean
      Random Forest    1.0000     1.0000  1.0000 1.000   0.9665
Logistic Regression    0.9722     0.9741  0.9722 0.972   0.9832
                SVM    0.9722     0.9741  0.9722 0.972   0.9833


## 5. Random Forest Hyperparameter Sensitivity

In [5]:
rf_rows = []
for n in [50, 100, 200, 300, 500]:
    model = RandomForestClassifier(n_estimators=n, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rf_rows.append({
        "n_estimators": n,
        "Test Accuracy": accuracy_score(y_test, pred),
        "Test F1": f1_score(y_test, pred, average="weighted"),
        "CV Mean": cross_val_score(model, X, y, cv=5, scoring="accuracy").mean()
    })

rf_sensitivity = pd.DataFrame(rf_rows)
display(rf_sensitivity.round(4))

 n_estimators  Test Accuracy  Test F1  CV Mean
           50            1.0      1.0   0.9610
          100            1.0      1.0   0.9721
          200            1.0      1.0   0.9665
          300            1.0      1.0   0.9665
          500            1.0      1.0   0.9663


## 6. SVM Hyperparameter Sensitivity

In [6]:
svm_rows = []
for C in [0.1, 0.5, 1, 2, 10]:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", C=C, gamma="scale"))
    ])
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    svm_rows.append({
        "C": C,
        "Test Accuracy": accuracy_score(y_test, pred),
        "Test F1": f1_score(y_test, pred, average="weighted"),
        "CV Mean": cross_val_score(model, X, y, cv=5, scoring="accuracy").mean()
    })

svm_sensitivity = pd.DataFrame(svm_rows)
display(svm_sensitivity.round(4))

   C  Test Accuracy  Test F1  CV Mean
 0.1         0.9722   0.9720   0.9663
 0.5         0.9722   0.9720   0.9833
 1.0         0.9722   0.9720   0.9833
 2.0         0.9722   0.9720   0.9889
10.0         0.9444   0.9432   0.9889


## 7. Final Evaluation

In [7]:
from sklearn.metrics import classification_report, confusion_matrix

best_name = results.iloc[0]["Model"]
best_model = models[best_name]
best_pred = best_model.predict(X_test)

print("Selected model:", best_name)
print(classification_report(y_test, best_pred, target_names=wine.target_names))
print("Confusion matrix:")
print(confusion_matrix(y_test, best_pred))

Selected model: Random Forest
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        12
     class_1       1.00      1.00      1.00        14
     class_2       1.00      1.00      1.00        10

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36

Confusion matrix:
[[12  0  0]
 [ 0 14  0]
 [ 0  0 10]]


## 8. Unsupervised K-Means and Silhouette Analysis

In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_scaled = StandardScaler().fit_transform(X)

cluster_rows = []
for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = model.fit_predict(X_scaled)
    cluster_rows.append({
        "k": k,
        "Silhouette": silhouette_score(X_scaled, labels),
        "Inertia": model.inertia_
    })

k_results = pd.DataFrame(cluster_rows)
best_k = int(k_results.loc[k_results["Silhouette"].idxmax(), "k"])
display(k_results.round(4))
print("Selected k:", best_k)

 k  Silhouette   Inertia
 2      0.2593 1658.7589
 3      0.2849 1277.9285
 4      0.2586 1175.3519
 5      0.2315 1107.0073
 6      0.2372 1046.0023
Selected k: 3


## 9. Critical Analysis

The high supervised score is valid for this experiment but does not prove production readiness. The dataset is small and relatively controlled. Five-fold cross-validation and hyperparameter sensitivity provide additional evidence, while independent external validation remains necessary.

K-Means is interpreted as unsupervised structure discovery. The target labels are not used to fit the clusters; they are only used after clustering for descriptive comparison.

### Recommendations
- Systematic cross-validated hyperparameter tuning
- Repeated cross-validation
- Independent external validation
- Class-specific error analysis
- Additional clustering algorithms
- Data-quality and drift monitoring

## 10. Reproducibility Checklist

- Public reproducible dataset loader
- Fixed random states
- Explicit cleaning and preprocessing
- Leakage-aware pipelines
- Multiple supervised models
- Multiple evaluation metrics
- Five-fold cross-validation
- Actual hyperparameter sensitivity experiments
- Unsupervised clustering with silhouette analysis
- Documented limitations and recommendations